In [ ]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier # O LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Fairness
from fairlearn.metrics import MetricFrame, selection_rate, demographic_parity_difference

# Explicabilidad
import shap

## 4.2 Modelo baseline

### 1. Construcción del Pipeline

In [ ]:
# 1. Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocesor),
    ('classifier', XGBClassifier(n_estimators=100, random_state=42))
])

# 2. Entrenamiento
pipeline.fit(X_train, y_train)

### 2. Rendimiento del modelo

In [ ]:
def evaluar_rendimiento(pipeline, X_test, y_test):
    """Calcula las métricas de rendimiento estándar del modelo."""
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_proba)
    }
    
    print("--- Rendimiento del Modelo ---")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")
    print("\nReporte de Clasificación:\n", classification_report(y_test, y_pred))
    
    return metrics

In [ ]:
metricas = evaluar_rendimiento(pipeline, X_test, y_test)

### 2. Evaluación de Equidad (Fairness)

In [ ]:
def evaluar_equidad(pipeline, X_test, y_test, grupo_sensible):
    """
    Evalúa si el modelo es 'justo' comparando predicciones entre grupos.
    grupo_sensible: Columna del dataset que representa la raza (atributo A).
    """
    y_pred = pipeline.predict(X_test)
    
    # Definimos las métricas a monitorizar por grupo
    metrics_dict = {
        'selection_rate': selection_rate, # Probabilidad de predecir reincidencia
        'accuracy': accuracy_score
    }
    
    mf = MetricFrame(
        metrics=metrics_dict,
        y_true=y_test,
        y_pred=y_pred,
        sensitive_features=grupo_sensible
    )
    
    dp_diff = demographic_parity_difference(y_test, y_pred, sensitive_features=grupo_sensible)
    
    print("--- Métricas de Equidad ---")
    print(f"Demographic Parity Difference: {dp_diff:.4f}")
    print("\nDesglose por grupo:\n", mf.by_group)
    
    return mf

In [ ]:
grupo_sensible = X_test["race"]  # o la columna sensible que uses
mf = evaluar_equidad(pipeline, X_test, y_test, grupo_sensible)

### 3. Explicabilidad con SHAP

In [ ]:
def explicar_modelo(pipeline, X_test):
    """Genera explicaciones post-hoc utilizando SHAP."""
    model = pipeline.named_steps['classifier']
    # Es necesario transformar los datos antes de pasarlos a SHAP
    X_transformed = pipeline.named_steps['preprocessor'].transform(X_test)
    
    # Nota: Si usas XGBoost o Random Forest, TreeExplainer es lo ideal
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_transformed)
    
    print("--- Generando Visualizaciones SHAP ---")
    # Resumen global de importancia de características
    shap.summary_plot(shap_values, X_transformed)

In [ ]:
explicar_modelo(pipeline, X_test)

## 4.3 Tratamiento de la problemática 1: Fairness